# KAGGLE_C00 Locked XAI Sample Manifest 128

Purpose: create the single fixed 128-case manifest used by every Grad-CAM and Guided Grad-CAM XAI notebook.

Add these Kaggle datasets to the notebook:

- `hintrngia/gate46-development-only-cxr`
- `hintrngia/gate7-test-seal`
- `hintrngia/cxr-lung-masks`

This notebook writes `/kaggle/working/locked_outputs/xai/xai_sample_manifest_128.csv`.


In [ ]:
import os
import json
import random
import hashlib
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

INPUT_DIR = Path('/kaggle/input')
WORK_DIR = Path('/kaggle/working')
OUT_DIR = WORK_DIR / 'locked_outputs' / 'xai'
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATASETS_REQUIRED = [
    'gate46-development-only-cxr',
    'gate7-test-seal',
    'cxr-lung-masks',
]

SAMPLE_SEED = 3407
N_PER_CLASS = 64


In [ ]:
def sha256_file(path, chunk_size=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()


def audit_required_datasets():
    found = {name: len(list(INPUT_DIR.rglob(name))) > 0 for name in DATASETS_REQUIRED}
    missing = [name for name, ok in found.items() if not ok]
    audit = {'required': DATASETS_REQUIRED, 'found': found, 'missing': missing}
    (OUT_DIR / 'xai_manifest_dataset_presence_audit.json').write_text(json.dumps(audit, indent=2), encoding='utf-8')
    if missing:
        raise FileNotFoundError(f'Missing required Kaggle datasets: {missing}')
    return audit


def extract_test_images_once():
    test_zip_matches = list(INPUT_DIR.rglob('opaque_test_images.zip'))
    test_root = WORK_DIR / 'opaque_test_images_extracted'
    if test_zip_matches:
        if not test_root.exists() or len(list(test_root.rglob('*.jpg'))) + len(list(test_root.rglob('*.jpeg'))) < 100:
            if test_root.exists():
                import shutil
                shutil.rmtree(test_root)
            test_root.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(test_zip_matches[0], 'r') as zf:
                zf.extractall(test_root)
    else:
        print('[AUDIT] opaque_test_images.zip not found; using images directly from /kaggle/input.')
    return test_root


def build_file_map(roots, include_masks=False):
    file_map = {}
    valid_exts = {'.jpg', '.jpeg', '.png'}
    for root in roots:
        for path in Path(root).rglob('*'):
            if not path.is_file() or path.suffix.lower() not in valid_exts:
                continue
            is_mask_path = 'mask' in path.name.lower() or 'mask' in path.parent.name.lower()
            if include_masks != is_mask_path:
                continue
            file_map[path.name] = path
    return file_map


def get_image_id(row):
    for col in ['image_id', 'image', 'filename', 'archive_member', 'source_relative_path']:
        if col in row.index and pd.notna(row[col]):
            return Path(str(row[col])).name
    raise KeyError('No image id/path column found in test label key.')


def get_label(row):
    if 'label_index' in row.index:
        return int(row['label_index'])
    if 'label' in row.index:
        value = row['label']
        if isinstance(value, str):
            return int(value.strip().upper() == 'PNEUMONIA')
        return int(value)
    raise KeyError('No label column found in test label key.')


def find_mask_for_image(image_name, mask_map):
    stem = Path(image_name).stem
    candidates = [
        f'{stem}.png',
        f'{stem}.jpg',
        f'{stem}.jpeg',
        f'{stem}_mask.png',
        f'{stem}_lung_mask.png',
        f'{stem}_lungs.png',
    ]
    for c in candidates:
        if c in mask_map:
            return mask_map[c]
    stem_lower = stem.lower()
    for name, path in mask_map.items():
        if Path(name).stem.lower().replace('_mask', '').replace('_lung', '').replace('_lungs', '') == stem_lower:
            return path
    return None


audit_required_datasets()
test_root = extract_test_images_once()
test_label_matches = list(INPUT_DIR.rglob('SEALED_test_label_key.csv'))
if not test_label_matches:
    raise FileNotFoundError('SEALED_test_label_key.csv not found.')
test_df = pd.read_csv(test_label_matches[0]).copy()
image_map = build_file_map([INPUT_DIR, test_root], include_masks=False)
mask_map = build_file_map([INPUT_DIR], include_masks=True)
print({'n_test_labels': len(test_df), 'n_images': len(image_map), 'n_masks': len(mask_map), 'label_key': str(test_label_matches[0])})


In [ ]:
rows = []
missing_images = []
missing_masks = []

for _, row in test_df.iterrows():
    image_id = get_image_id(row)
    label = get_label(row)
    image_path = image_map.get(image_id)
    mask_path = find_mask_for_image(image_id, mask_map)
    if image_path is None:
        missing_images.append(image_id)
        continue
    if mask_path is None:
        missing_masks.append(image_id)
        continue
    rows.append({
        'image_id': image_id,
        'image_path_or_archive_member': str(image_path),
        'image_sha256': sha256_file(image_path),
        'mask_id': mask_path.name,
        'mask_path': str(mask_path),
        'mask_sha256': sha256_file(mask_path),
        'true_label': int(label),
    })

eligible = pd.DataFrame(rows)
audit = {
    'n_test_labels': int(len(test_df)),
    'n_eligible_with_image_and_mask': int(len(eligible)),
    'n_missing_images': int(len(missing_images)),
    'n_missing_masks': int(len(missing_masks)),
    'missing_images_first_20': missing_images[:20],
    'missing_masks_first_20': missing_masks[:20],
}
(OUT_DIR / 'xai_manifest_eligibility_audit.json').write_text(json.dumps(audit, indent=2), encoding='utf-8')
print(json.dumps(audit, indent=2))

if len(eligible) < 128:
    raise RuntimeError('Fewer than 128 test cases have both image and lung mask.')

counts = eligible['true_label'].value_counts().to_dict()
if counts.get(0, 0) < N_PER_CLASS or counts.get(1, 0) < N_PER_CLASS:
    raise RuntimeError(f'Not enough cases per class for balanced 64/64 sample: {counts}')

sampled = (
    eligible.groupby('true_label', group_keys=False)
    .apply(lambda g: g.sample(n=N_PER_CLASS, random_state=SAMPLE_SEED))
    .sort_values(['true_label', 'image_id'])
    .reset_index(drop=True)
)
sampled['sampling_stratum'] = sampled['true_label'].map({0: 'normal', 1: 'pneumonia'})
sampled['sample_index'] = np.arange(len(sampled))
sampled['sample_seed'] = SAMPLE_SEED
sampled['manifest_id'] = 'xai_sample_manifest_128_seed3407_balanced_64_64'

ordered_cols = [
    'manifest_id', 'sample_seed', 'sample_index', 'sampling_stratum',
    'image_id', 'image_path_or_archive_member', 'image_sha256',
    'mask_id', 'mask_path', 'mask_sha256', 'true_label'
]
sampled = sampled[ordered_cols]
out_path = OUT_DIR / 'xai_sample_manifest_128.csv'
sampled.to_csv(out_path, index=False)

manifest_summary = {
    'manifest_file': str(out_path),
    'n_rows': int(len(sampled)),
    'class_counts': sampled['true_label'].value_counts().sort_index().astype(int).to_dict(),
    'sha256': sha256_file(out_path),
    'rule': 'Balanced deterministic sample: 64 normal and 64 pneumonia, seed 3407; independent of model correctness/confidence.',
}
(OUT_DIR / 'xai_sample_manifest_128_summary.json').write_text(json.dumps(manifest_summary, indent=2), encoding='utf-8')
print(json.dumps(manifest_summary, indent=2))
sampled.head()
